# Huấn luyện CNN bằng TensorFlow / Keras

Notebook này dùng **cùng dataset** với các notebook còn lại: `dataset_raw/animals/animals`, có cấu trúc `class_name/image_file`.

In [ ]:
from pathlib import Path
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATASET_DIR = Path('dataset_raw/animals/animals')
assert DATASET_DIR.exists(), f'Không tìm thấy {DATASET_DIR.resolve()}'
print('Dataset chung:', DATASET_DIR.resolve())

## 1. Nạp dataset chung

Tách 80% train và 20% validation bằng cùng seed để tái lập kết quả.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=.2, subset='training', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
valid_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=.2, subset='validation', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
class_names = list(train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds, valid_ds = train_ds.prefetch(AUTOTUNE), valid_ds.prefetch(AUTOTUNE)
print(f'{len(class_names)} lớp:', class_names)

## 2. Định nghĩa CNN

`Rescaling` chuẩn hoá pixel về [0, 1]; ba khối Conv2D học đặc trưng từ cạnh đến hình dạng. `last_conv` được đặt tên để dùng Grad-CAM ở service FastAPI.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((224, 224, 3)),
    tf.keras.layers.Rescaling(1 / 255.0),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', name='last_conv'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(.25),
    tf.keras.layers.Dense(len(class_names), activation='softmax'),
], name='animal_cnn_keras')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
history = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS, callbacks=callbacks)
model.save('animal_image_classifier.keras')
print('Đã lưu animal_image_classifier.keras')

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(); plt.show()